In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as f
import torchvision
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
# CIFAR-10 이미지 데이터 활용
# Horizontal flip 적용(train data)

data_dir = "./data"
batch_size = 128

cifar_mean = [0.4914, 0.4822, 0.4465]

# train data -> 32 * 32 size에 padding 4를 적용하고 랜덤하게 32 * 32 크기로 사진을 잘라서 사용함
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=[0.2470, 0.2435, 0.2616]),
])

# test data -> 사진을 그대로 사용
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=[0.2470, 0.2435, 0.2616]),
])


In [ ]:
train_dataset = datasets.CIFAR10(root=data_dir, train=True, download=False, transform=train_transform)
val_dataset = datasets.CIFAR10(root=data_dir, train=False, download=False, transform=val_transform)

'''
batch_size : 모델에 한번에 얼마나 많은 데이터를 넣을지
shuffle : 각 epoch마다 데이터 순서를 바꿈
pin_memory : pinned memory를 사용해 gpu로 데이터 빠르게 복사
'''
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

### Resnet(CIFAR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Option A
# 채널이 달라지는 구간에서 논문의 Option A 방법 사용
# Option A : 부족한 채널의 깊이를 0으로 padding 해 shortcut의 채널의 깊이를 맞춘다.
class Shortcut_cls(nn.Module) :
    def __init__(self, cin, cout) :
        super().__init__()
        self.padded = cout - cin

    def forward(self, x) :
        out = x
        out = out[:, :, ::2, ::2]
        b, _, h, w = out.shape
        zero_padding = torch.zeros(b, self.padded, h, w,
                                    dtype = out.dtype,
                                    device = out.device)
        # torch.zeros 는 cpu에서 연산을 수행하고 모델 학습은 GPU에서 수행될 시 메모리 공간이 달라서 에러 발생
        out = torch.cat([out, zero_padding],
                        dim = 1 ) # 채널기준으로 합침
        return out


# 동일차원 출력
class BasicBlock(nn.Module):
    """
    ResNet basic block (CIFAR style):
    F(x) = conv3x3 -> BN -> ReLU -> conv3x3 -> BN
    y = ReLU( F(x) + x ) : identity shortcut 사용
    """
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        # batch normalization을 사용하므로 conv bias = False 설정
        # 처음 feature가 줄어들 때 stride를 2로 설정
        self.conv1 = nn.Conv2d(in_channels=cin,
                                out_channels= cout,
                                kernel_size=3,
                                stride=stride,
                                padding=1,
                                bias = False )
        self.bn1 = nn.BatchNorm2d(cout)

        self.conv2 = nn.Conv2d(in_channels=cout,
                                out_channels= cout,
                                kernel_size=3,
                                stride=1,
                                padding=1,
                                bias = False )
        self.bn2 = nn.BatchNorm2d(cout)

        self.relu = nn.ReLU()
        if cin != cout or stride == 2:
            self.shortcut = Shortcut_cls(cin, cout)
        else :
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + self.shortcut(x)
        out = self.relu(out)

        return out

class CustomResNet(nn.Module):
    '''
    3 * 32 * 32 -> 16 * 32 * 32
    16  * 32 * 32 -> 32 * 16 * 16
    32 * 16 * 16 -> 64 * 8 * 8
    FCL
    '''
    def __init__(self) :
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels= 3,
            out_channels= 16,
            stride = 1,
            kernel_size= 3,
            padding = 1,
            bias = False
            )
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.layer1 = nn.Sequential(
            BasicBlock(16, 16, stride = 1),
            BasicBlock(16, 16, stride = 1),
            BasicBlock(16, 16, stride = 1),
            BasicBlock(16, 16, stride = 1),
            BasicBlock(16, 16, stride = 1),
        )
        self.layer2 = nn.Sequential(
            BasicBlock(16, 32, stride = 2),
            BasicBlock(32, 32, stride = 1),
            BasicBlock(32, 32, stride = 1),
            BasicBlock(32, 32, stride = 1),
            BasicBlock(32, 32, stride = 1),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(32, 64, stride = 2),
            BasicBlock(64, 64, stride = 1),
            BasicBlock(64, 64, stride = 1),
            BasicBlock(64, 64, stride = 1),
            BasicBlock(64, 64, stride = 1),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fcl = nn.Linear(64, 10)


    def forward(self, x) :
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.gap(out)
        out = torch.flatten(out, 1)
        out = self.fcl(out)
        return out


